# Experiment: Plate Raster Synchrony Viewer

Objective:
- Discover analyzed scan outputs under the `.../Network/<scan_id>/wellXYZ/` pattern.
- Render one interactive 4x6 plate view per scan with raster, synchrony, or both overlaid in each well.
- Export publication-style PNGs at configurable DPI plus interactive HTML for review.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import plotly.graph_objects as go
from plotly.subplots import make_subplots


## Notes

- Expected analyzed layout: `<root>/.../Network/<scan_id>/wellXYZ/`
- Plot inputs per well: `spike_times.npy` and `network_results.json`
- Plate mapping is fixed to row-major order: `well000 -> A1`, `well001 -> A2`, ..., `well023 -> D6`
- Plotly provides zoom/pan/scroll in notebook output. PNG export uses Plotly image export and typically requires `kaleido`.


In [ ]:
PLATE_ROWS = 4
PLATE_COLS = 6
PLATE_WELL_COUNT = PLATE_ROWS * PLATE_COLS
ROW_LABELS = ['A', 'B', 'C', 'D']
DISPLAY_MODES = {'raster', 'synchrony', 'both'}


def well_to_plate_position(well_id: str) -> tuple[int, int, str]:
    well_num = int(str(well_id).replace('well', ''))
    if not 0 <= well_num < PLATE_WELL_COUNT:
        raise ValueError(f'Well {well_id} is outside the supported 24-well range.')
    row = well_num // PLATE_COLS
    col = well_num % PLATE_COLS
    label = f"{ROW_LABELS[row]}{col + 1}"
    return row + 1, col + 1, label


def scan_context_label(scan_dir: Path) -> str:
    parts = scan_dir.parts
    try:
        network_idx = parts.index('Network')
    except ValueError:
        return scan_dir.as_posix()
    start_idx = max(0, network_idx - 3)
    return '/'.join(parts[start_idx:network_idx + 2])


def subplot_axis_ref(row: int, col: int) -> tuple[str, str]:
    axis_index = (row - 1) * PLATE_COLS + col
    suffix = '' if axis_index == 1 else str(axis_index)
    return f'x{suffix} domain', f'y{suffix} domain'


def discover_well_records(root_dir: str | Path) -> pd.DataFrame:
    root = Path(root_dir).expanduser().resolve()
    rows: list[dict] = []

    for well_dir in root.rglob('well*'):
        if not well_dir.is_dir() or not well_dir.name.startswith('well'):
            continue
        scan_dir = well_dir.parent
        if scan_dir.parent.name != 'Network':
            continue
        try:
            row_idx, col_idx, plate_label = well_to_plate_position(well_dir.name)
        except ValueError:
            continue
        spike_path = well_dir / 'spike_times.npy'
        network_json = well_dir / 'network_results.json'
        rows.append({
            'root_dir': str(root),
            'scan_dir': str(scan_dir.resolve()),
            'scan_id': scan_dir.name,
            'scan_label': scan_context_label(scan_dir.resolve()),
            'well_id': well_dir.name,
            'plate_label': plate_label,
            'row': row_idx,
            'col': col_idx,
            'spike_times_path': str(spike_path),
            'network_json_path': str(network_json),
            'has_spike_times': spike_path.exists(),
            'has_network_json': network_json.exists(),
        })

    if not rows:
        return pd.DataFrame(columns=[
            'root_dir', 'scan_dir', 'scan_id', 'scan_label', 'well_id', 'plate_label',
            'row', 'col', 'spike_times_path', 'network_json_path', 'has_spike_times', 'has_network_json'
        ])

    df = pd.DataFrame(rows).drop_duplicates(subset=['scan_dir', 'well_id']).sort_values(['scan_dir', 'well_id'])
    return df.reset_index(drop=True)


def summarize_scans(index_df: pd.DataFrame) -> pd.DataFrame:
    if index_df.empty:
        return pd.DataFrame(columns=['scan_label', 'scan_dir', 'n_wells', 'missing_spike_times', 'missing_network_json'])
    summary = (
        index_df.groupby(['scan_label', 'scan_dir'], as_index=False)
        .agg(
            n_wells=('well_id', 'nunique'),
            missing_spike_times=('has_spike_times', lambda s: int((~s).sum())),
            missing_network_json=('has_network_json', lambda s: int((~s).sum())),
        )
        .sort_values('scan_dir')
        .reset_index(drop=True)
    )
    return summary


In [ ]:
def load_spike_times(spike_path: str | Path) -> dict:
    data = np.load(Path(spike_path), allow_pickle=True).item()
    return {k: np.asarray(v, dtype=float) for k, v in data.items()}


def load_network_plot_data(network_json_path: str | Path) -> dict:
    with open(network_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    plot_data = data.get('plot_data', {})
    for key in ('t', 'signal', 'signal_smooth', 'burst_peak_times', 'burst_peak_values'):
        if key in plot_data and plot_data[key] is not None:
            plot_data[key] = np.asarray(plot_data[key], dtype=float)
    return plot_data


def sort_units(spike_times: dict, mode: str = 'firing_rate_desc') -> list:
    if mode == 'native':
        return list(spike_times.keys())

    def last_time(arr: np.ndarray) -> float:
        return float(arr[-1]) if arr.size else 0.0

    if mode == 'firing_rate_desc':
        duration = max((last_time(v) for v in spike_times.values()), default=1.0) or 1.0
        return sorted(spike_times.keys(), key=lambda u: len(spike_times[u]) / duration, reverse=True)

    return sorted(spike_times.keys(), key=lambda u: str(u))


def raster_traces(spike_times: dict, well_name: str, mode: str = 'firing_rate_desc', marker_size: float = 5.0) -> tuple[list[go.Scattergl], int, float]:
    traces: list[go.Scattergl] = []
    units = sort_units(spike_times, mode=mode)
    max_time = 0.0
    for unit_index, unit in enumerate(units, start=1):
        unit_spikes = np.asarray(spike_times[unit], dtype=float)
        if unit_spikes.size == 0:
            continue
        max_time = max(max_time, float(unit_spikes[-1]))
        traces.append(
            go.Scattergl(
                x=unit_spikes,
                y=np.full(unit_spikes.shape, unit_index, dtype=float),
                mode='markers',
                marker=dict(symbol='line-ns-open', size=marker_size, color='rgba(90, 90, 90, 0.75)'),
                name='Raster',
                legendgroup='raster',
                showlegend=False,
                hovertemplate=f'{well_name}<br>Unit {unit}<br>t=%{{x:.3f}} s<extra></extra>',
            )
        )
    return traces, max(len(units), 1), max_time


def synchrony_traces(plot_data: dict, well_name: str, line_width: float = 1.25) -> tuple[list[go.Scattergl], float, float]:
    traces: list[go.Scattergl] = []
    t = np.asarray(plot_data.get('t', []), dtype=float)
    signal = np.asarray(plot_data.get('signal', []), dtype=float)
    xmax = float(t[-1]) if t.size else 0.0
    ymax = float(np.nanmax(signal)) if signal.size else 0.0
    if t.size and signal.size:
        traces.append(
            go.Scattergl(
                x=t,
                y=signal,
                mode='lines',
                line=dict(color='#b22222', width=line_width),
                name='Synchrony',
                legendgroup='synchrony',
                showlegend=False,
                hovertemplate=f'{well_name}<br>Synchrony=%{{y:.3f}}<br>t=%{{x:.3f}} s<extra></extra>',
            )
        )
    smooth = plot_data.get('signal_smooth')
    if smooth is not None:
        smooth = np.asarray(smooth, dtype=float)
        if t.size and smooth.size:
            ymax = max(ymax, float(np.nanmax(smooth)))
            traces.append(
                go.Scattergl(
                    x=t,
                    y=smooth,
                    mode='lines',
                    line=dict(color='rgba(255, 140, 0, 0.95)', width=max(1.0, line_width - 0.1)),
                    name='Synchrony smooth',
                    legendgroup='synchrony',
                    showlegend=False,
                    hovertemplate=f'{well_name}<br>Smooth=%{{y:.3f}}<br>t=%{{x:.3f}} s<extra></extra>',
                )
            )
    peak_t = np.asarray(plot_data.get('burst_peak_times', []), dtype=float)
    peak_y = np.asarray(plot_data.get('burst_peak_values', []), dtype=float)
    if peak_t.size and peak_y.size:
        ymax = max(ymax, float(np.nanmax(peak_y)))
        traces.append(
            go.Scattergl(
                x=peak_t,
                y=peak_y,
                mode='markers',
                marker=dict(color='red', size=5, symbol='circle'),
                name='Burst peaks',
                legendgroup='synchrony',
                showlegend=False,
                hovertemplate=f'{well_name}<br>Burst peak=%{{y:.3f}}<br>t=%{{x:.3f}} s<extra></extra>',
            )
        )
    baseline = plot_data.get('baseline')
    threshold = plot_data.get('threshold')
    if baseline is not None and t.size:
        ymax = max(ymax, float(baseline))
        traces.append(
            go.Scattergl(
                x=t,
                y=np.full(t.shape, float(baseline)),
                mode='lines',
                line=dict(color='rgba(255, 102, 0, 0.7)', width=1, dash='dash'),
                name='Baseline',
                legendgroup='synchrony',
                showlegend=False,
                hoverinfo='skip',
            )
        )
    if threshold is not None and t.size:
        ymax = max(ymax, float(threshold))
        traces.append(
            go.Scattergl(
                x=t,
                y=np.full(t.shape, float(threshold)),
                mode='lines',
                line=dict(color='rgba(192, 57, 43, 0.8)', width=1, dash='dash'),
                name='Threshold',
                legendgroup='synchrony',
                showlegend=False,
                hoverinfo='skip',
            )
        )
    return traces, ymax, xmax


In [ ]:
def create_scan_figure(
    index_df: pd.DataFrame,
    scan_dir: str | Path,
    *,
    display_mode: str = 'both',
    marker_size: float = 5.0,
    line_width: float = 1.25,
    width_px: int = 3600,
    height_px: int = 2400,
    unit_sort_mode: str = 'firing_rate_desc',
    title: str | None = None,
) -> go.Figure:
    if display_mode not in DISPLAY_MODES:
        raise ValueError(f'display_mode must be one of {DISPLAY_MODES}')

    scan_dir = str(Path(scan_dir).resolve())
    scan_df = index_df[index_df['scan_dir'] == scan_dir].copy()
    if scan_df.empty:
        raise ValueError(f'No wells found for scan: {scan_dir}')

    subplot_titles = []
    for well_num in range(PLATE_WELL_COUNT):
        row, col, plate_label = well_to_plate_position(f'well{well_num:03d}')
        subplot_titles.append(f'{plate_label}<br>well{well_num:03d}')

    fig = make_subplots(
        rows=PLATE_ROWS,
        cols=PLATE_COLS,
        specs=[[{'secondary_y': True} for _ in range(PLATE_COLS)] for _ in range(PLATE_ROWS)],
        subplot_titles=subplot_titles,
        horizontal_spacing=0.02,
        vertical_spacing=0.07,
    )

    trace_roles: list[str] = []
    trace_showlegend: dict[str, bool] = {'raster': True, 'synchrony': True}
    global_xmax = 0.0

    for well_num in range(PLATE_WELL_COUNT):
        well_id = f'well{well_num:03d}'
        row, col, _ = well_to_plate_position(well_id)
        matches = scan_df[scan_df['well_id'] == well_id]
        xref, yref = subplot_axis_ref(row, col)

        fig.update_xaxes(title_text='Time (s)' if row == PLATE_ROWS else None, row=row, col=col)
        fig.update_yaxes(title_text='Unit' if col == 1 else None, row=row, col=col, secondary_y=False)
        fig.update_yaxes(title_text='Sync' if col == PLATE_COLS else None, row=row, col=col, secondary_y=True)

        if matches.empty:
            fig.add_annotation(x=0.5, y=0.5, xref=xref, yref=yref, text='Missing well', showarrow=False, font=dict(size=12, color='gray'))
            continue

        record = matches.iloc[0]
        raster_loaded = False
        sync_loaded = False

        if bool(record['has_spike_times']):
            spike_times = load_spike_times(record['spike_times_path'])
            raster, unit_count, spike_xmax = raster_traces(spike_times, well_name=well_id, mode=unit_sort_mode, marker_size=marker_size)
            fig.update_yaxes(range=[0, unit_count + 1], row=row, col=col, secondary_y=False)
            global_xmax = max(global_xmax, spike_xmax)
            for trace in raster:
                trace.showlegend = trace_showlegend['raster']
                trace_showlegend['raster'] = False
                fig.add_trace(trace, row=row, col=col, secondary_y=False)
                trace_roles.append('raster')
            raster_loaded = True

        if bool(record['has_network_json']):
            plot_data = load_network_plot_data(record['network_json_path'])
            sync, sync_ymax, sync_xmax = synchrony_traces(plot_data, well_name=well_id, line_width=line_width)
            global_xmax = max(global_xmax, sync_xmax)
            if sync_ymax > 0:
                fig.update_yaxes(range=[0, sync_ymax * 1.1], row=row, col=col, secondary_y=True)
            for trace in sync:
                trace.showlegend = trace_showlegend['synchrony']
                trace_showlegend['synchrony'] = False
                fig.add_trace(trace, row=row, col=col, secondary_y=True)
                trace_roles.append('synchrony')
            sync_loaded = True

        if not raster_loaded and not sync_loaded:
            fig.add_annotation(x=0.5, y=0.5, xref=xref, yref=yref, text='No plot data', showarrow=False, font=dict(size=12, color='gray'))
        elif not raster_loaded or not sync_loaded:
            missing_label = 'raster missing' if not raster_loaded else 'synchrony missing'
            fig.add_annotation(
                x=0.98,
                y=0.94,
                xref=xref,
                yref=yref,
                text=missing_label,
                showarrow=False,
                xanchor='right',
                font=dict(size=10, color='gray'),
            )

    if global_xmax > 0:
        fig.update_xaxes(range=[0, global_xmax])

    mode_visibility = {
        'both': [True for _ in trace_roles],
        'raster': [role == 'raster' for role in trace_roles],
        'synchrony': [role == 'synchrony' for role in trace_roles],
    }
    for trace, visible in zip(fig.data, mode_visibility[display_mode]):
        trace.visible = visible

    label = title or scan_df['scan_label'].iloc[0]
    fig.update_layout(
        width=width_px,
        height=height_px,
        title=dict(text=f'Plate Raster + Synchrony: {label}', x=0.5),
        template='plotly_white',
        hovermode='closest',
        margin=dict(l=40, r=40, t=90, b=50),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        updatemenus=[
            dict(
                type='buttons',
                direction='left',
                x=0.5,
                y=1.12,
                xanchor='center',
                yanchor='top',
                buttons=[
                    dict(label='Both', method='update', args=[{'visible': mode_visibility['both']}]),
                    dict(label='Raster only', method='update', args=[{'visible': mode_visibility['raster']}]),
                    dict(label='Synchrony only', method='update', args=[{'visible': mode_visibility['synchrony']}]),
                ],
            )
        ],
    )
    return fig


In [ ]:
def export_scan_figure(
    fig: go.Figure,
    *,
    output_dir: str | Path,
    stem: str,
    export_png: bool = True,
    export_html: bool = True,
    dpi: int = 600,
    width_in: float = 24.0,
    height_in: float = 16.0,
) -> dict[str, Path]:
    out_dir = Path(output_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    exported: dict[str, Path] = {}

    width_px = int(width_in * dpi)
    height_px = int(height_in * dpi)

    if export_html:
        html_path = out_dir / f'{stem}.html'
        fig.write_html(html_path, include_plotlyjs='cdn')
        exported['html'] = html_path

    if export_png:
        png_path = out_dir / f'{stem}.png'
        try:
            fig.write_image(png_path, format='png', width=width_px, height=height_px, scale=1)
        except Exception as exc:
            raise RuntimeError('PNG export requires Plotly image export support, typically via `pip install kaleido`.') from exc
        exported['png'] = png_path

    return exported


def export_all_scans(
    index_df: pd.DataFrame,
    *,
    output_dir: str | Path,
    display_mode: str = 'both',
    dpi: int = 600,
    width_in: float = 24.0,
    height_in: float = 16.0,
    marker_size: float = 5.0,
    line_width: float = 1.25,
    unit_sort_mode: str = 'firing_rate_desc',
) -> list[dict]:
    exports: list[dict] = []
    export_width_px = int(width_in * dpi)
    export_height_px = int(height_in * dpi)
    for scan_dir in index_df['scan_dir'].drop_duplicates().tolist():
        fig = create_scan_figure(
            index_df,
            scan_dir,
            display_mode=display_mode,
            marker_size=marker_size,
            line_width=line_width,
            width_px=export_width_px,
            height_px=export_height_px,
            unit_sort_mode=unit_sort_mode,
        )
        stem = Path(scan_dir).name + '_plate_overlay'
        exported = export_scan_figure(
            fig,
            output_dir=output_dir,
            stem=stem,
            dpi=dpi,
            width_in=width_in,
            height_in=height_in,
        )
        exports.append({'scan_dir': scan_dir, **{k: str(v) for k, v in exported.items()}})
    return exports


## Configure paths and rendering

Set `ANALYSIS_ROOT` to the analyzed data directory that contains `.../Network/<scan_id>/wellXYZ/` outputs.


In [ ]:
ANALYSIS_ROOT = Path('/path/to/analyzed/data')
OUTPUT_DIR = Path('./plate_exports')

DISPLAY_MODE = 'both'          # 'raster', 'synchrony', or 'both'
EXPORT_DPI = 600               # user-configurable static export DPI
FIGURE_WIDTH_IN = 24.0         # publication/export width in inches
FIGURE_HEIGHT_IN = 16.0        # publication/export height in inches
PREVIEW_WIDTH_PX = 3600        # interactive notebook display width
PREVIEW_HEIGHT_PX = 2400       # interactive notebook display height
MARKER_SIZE = 5.0
LINE_WIDTH = 1.25
UNIT_SORT_MODE = 'firing_rate_desc'

EXPORT_HTML = True
EXPORT_PNG = True


In [ ]:
index_df = discover_well_records(ANALYSIS_ROOT)
summary_df = summarize_scans(index_df)
print(f'Found {len(summary_df)} scan(s) and {len(index_df)} well record(s).')
display(summary_df)


## Preview one scan interactively

Choose a scan index from `summary_df` and render it below. Use Plotly tools to zoom, pan, and inspect individual wells.


In [ ]:
SCAN_INDEX = 0

if summary_df.empty:
    raise RuntimeError('No scans found. Check ANALYSIS_ROOT and the expected directory pattern.')

selected_scan_dir = summary_df.iloc[SCAN_INDEX]['scan_dir']
fig = create_scan_figure(
    index_df,
    selected_scan_dir,
    display_mode=DISPLAY_MODE,
    marker_size=MARKER_SIZE,
    line_width=LINE_WIDTH,
    width_px=PREVIEW_WIDTH_PX,
    height_px=PREVIEW_HEIGHT_PX,
    unit_sort_mode=UNIT_SORT_MODE,
)
fig.show()


## Export current scan or all scans

- `export_scan_figure(...)` saves the current interactive figure as HTML and/or PNG.
- `export_all_scans(...)` renders and exports every discovered scan using the current configuration.


In [ ]:
CURRENT_STEM = Path(selected_scan_dir).name + '_plate_overlay'

# Export the currently displayed scan.
# exported_paths = export_scan_figure(
#     fig,
#     output_dir=OUTPUT_DIR,
#     stem=CURRENT_STEM,
#     export_png=EXPORT_PNG,
#     export_html=EXPORT_HTML,
#     dpi=EXPORT_DPI,
#     width_in=FIGURE_WIDTH_IN,
#     height_in=FIGURE_HEIGHT_IN,
# )
# exported_paths

# Export every discovered scan.
# all_exports = export_all_scans(
#     index_df,
#     output_dir=OUTPUT_DIR,
#     display_mode=DISPLAY_MODE,
#     dpi=EXPORT_DPI,
#     width_in=FIGURE_WIDTH_IN,
#     height_in=FIGURE_HEIGHT_IN,
#     marker_size=MARKER_SIZE,
#     line_width=LINE_WIDTH,
#     unit_sort_mode=UNIT_SORT_MODE,
# )
# pd.DataFrame(all_exports)
